In [1]:
import pandas as pd

In [2]:
datos = pd.read_csv("xCh.csv")

In [3]:
datos.head()

,Unnamed: 0,Angle,PSD
0,0,5.00000,1980
1,1,5.04003,1924
2,2,5.08006,1869
3,3,5.12010,1939
4,4,5.16013,1780


In [4]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1625 entries, 0 to 1624
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  1625 non-null   int64  
 1        Angle  1625 non-null   float64
 2          PSD  1625 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 38.2 KB


In [10]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import pathlib
import webbrowser

csv_file     = "xCh.csv"
sample_label = "Café de Chiapas"
html_output  = "xCh_difractograma.html"

# ======= 1. Leer y preparar datos =======
datos = pd.read_csv(csv_file)
df = datos.copy()
df.columns = df.columns.str.strip()
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# ======= 2. Config común para imágenes HD =======
def make_config(nombre_png):
    return dict(
        displaylogo=False,  # oculta logo de Plotly
        toImageButtonOptions=dict(
            format="png",
            filename=nombre_png,
            height=1200,   # más alto
            width=2000,    # más ancho
            scale=3        # más resolución (3x)
        )
    )

# ======= 3. Función para figura de región =======
def make_region_fig(df, tmin, tmax, titulo_extra=""):
    mask = (df["Angle"] >= tmin) & (df["Angle"] <= tmax)
    df_zoom = df.loc[mask].copy()

    titulo_base = f"Región {tmin:.0f}°–{tmax:.0f}°"
    titulo = f"{titulo_base} – {titulo_extra}" if titulo_extra else titulo_base

    fig = px.line(
        df_zoom,
        x="Angle",
        y="PSD",
        title=titulo,
        labels={"Angle": "2θ (°)", "PSD": "Intensidad (cuentas PSD)"}
    )

    fig.update_traces(hovertemplate="2θ = %{x:.2f}°<br>I = %{y:.0f}")
    fig.update_layout(
        width=900,
        height=400,
        margin=dict(l=40, r=40, t=60, b=40)
    )
    return fig

# ======= 4. Figura TOTAL =======
fig_total = px.line(
    df,
    x="Angle",
    y="PSD",
    title=f"Difractograma de rayos X – {sample_label} (interactivo)",
    labels={"Angle": "2θ (°)", "PSD": "Intensidad (cuentas PSD)"}
)
fig_total.update_traces(hovertemplate="2θ = %{x:.2f}°<br>I = %{y:.0f}")
fig_total.update_xaxes(dtick=5)
fig_total.update_layout(
    width=900,
    height=500,
    margin=dict(l=40, r=40, t=60, b=40)
)

# ======= 5. Figuras de regiones =======
fig_r1 = make_region_fig(df, 5, 18)
fig_r2 = make_region_fig(df, 20, 35)
fig_r3 = make_region_fig(df, 35, 45)

# ======= 6. Convertir figuras a HTML embebido con CONFIG HD =======
html_total = pio.to_html(
    fig_total,
    full_html=False,
    include_plotlyjs="cdn",
    config=make_config("xCh_total")
)

html_r1 = pio.to_html(
    fig_r1,
    full_html=False,
    include_plotlyjs=False,
    config=make_config("xCh_region_5_18")
)

html_r2 = pio.to_html(
    fig_r2,
    full_html=False,
    include_plotlyjs=False,
    config=make_config("xCh_region_20_35")
)

html_r3 = pio.to_html(
    fig_r3,
    full_html=False,
    include_plotlyjs=False,
    config=make_config("xCh_region_35_45")
)

# ======= 7. Construir la página HTML =======
html_page = f"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8">
  <title>Difractogramas de rayos X – {sample_label}</title>
  <style>
    body {{
      margin: 0;
      padding: 2rem;
      font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      background: #020617;
      color: #e5e7eb;
    }}
    h1 {{
      text-align: center;
      margin-bottom: 2rem;
    }}
    h2 {{
      margin-top: 0;
      text-align: center;
      color: #e5e7eb;
    }}
    .section {{
      background: #0f172a;
      border-radius: 16px;
      padding: 1.5rem;
      margin: 0 auto 2rem auto;
      max-width: 1100px;
      box-shadow: 0 18px 45px rgba(0,0,0,0.5);
    }}
  </style>
</head>
<body>
  <h1>Difractogramas de rayos X – {sample_label}</h1>

  <div class="section">
    <h2>Difractograma completo</h2>
    {html_total}
  </div>

  <div class="section">
    <h2>Región 5°–18°</h2>
    {html_r1}
  </div>

  <div class="section">
    <h2>Región 20°–35° s</h2>
    {html_r2}
  </div>

  <div class="section">
    <h2>Región 35°–45° </h2>
    {html_r3}
  </div>
</body>
</html>
"""

out_path = pathlib.Path(html_output)
out_path.write_text(html_page, encoding="utf-8")
webbrowser.open(out_path.resolve().as_uri())
print(f"Página generada: {out_path}")


Página generada: xCh_difractograma.html


Opening in existing browser session.
